# 05 Interpretation & Limitations

## What the results support

The lightweight M metric captures pair-specific activation-side structural information.

Its matched mean alignment is +0.0420, compared with a permutation-null mean of about -0.0021, with an empirical one-sided p-value of about 0.002.

L1 is much more consistent. Once the correct first token is supplied, continuation moves in the expected direction on all 14 held-out pairs.

The M result is also heterogeneous across relation types. Role/argument binding, modifier attachment, and spatial relations are consistently positive, comparative relations are negative, and quantifier binding is mixed.

## What the results do not establish

The results do not show that activation-side information is unnecessary. M still shows a real matched signal beyond its permutation null.

They also do not show that full multi-token J-Lens is weaker than L1. M is only a lightweight structural alignment metric, not a fully calibrated multi-token readout.

The negative comparative results do not show that comparative information is absent from the model. They may reflect limitations of the readout, calibration, source representation, or geometric mismatch.

The experiment also does not support broad claims across models or tasks. It uses one model, a small controlled dataset, and a limited late-layer range.

Finally, M and L1 cannot be turned into a percentage decomposition. They use different score spaces, and continuation itself is produced through the model's internal representations.

## Main limitations

The main limitations are:

1. M is a lightweight structural metric rather than a full calibrated multi-token J-Lens readout.
2. The held-out dataset contains only 14 pairs.
3. Source and candidate formulations are hand-designed.
4. Source representation uses only the final-token residual state.
5. Only a selected late-layer range is evaluated.
6. L1 receives the correct first token, making it a deliberately strong baseline.
7. M and L1 are compared using directional success rather than a shared quantitative score.
8. The experiment does not identify why some relation types, especially comparative relations, are harder for M.

## Alternative explanations

Several factors could contribute to the observed pattern.

The fixed carrier prompt may influence the candidate concept vectors. The final source token may also be an incomplete location for reading relational structure.

Comparative relations may use a geometry that is poorly captured by simple source-difference versus concept-difference cosine alignment.

L1 may benefit strongly from ordinary lexical and semantic continuation priors once the first token is known.

The categories may also differ in intrinsic difficulty, and with only a few examples per category, some apparent category effects may be item-specific.

These possibilities are not resolved by the current experiment.

## Stronger follow-up

The clearest next step would be to replace lightweight M with a more fully calibrated multi-token readout while keeping the same held-out structural evaluation.

I would keep the same source contexts, candidates, L1 baseline, and permutation controls, but improve the activation-side readout with stronger calibration, normalization, and representation matching.

The most informative result would be an M-only case: a held-out pair where the improved activation-side readout succeeds, L1 fails, and the M signal survives its null control.

Comparative relations would be especially useful for diagnosing whether the current failure comes from calibration, representation geometry, or the readout design itself.

## 5.6 — Final Research Conclusion

This project asked whether multi-token recovery from J-Lens-style readouts is
primarily driven by activation-derived information, or whether much of the
apparent recovery can already be explained by ordinary language-model
continuation once the first token is known.

The results support a mixed answer.

The lightweight activation-conditioned multi-token estimator contains genuine,
pair-specific structural information. Its matched source/concept structural
alignment is significantly higher than a pair-permutation null
(empirical p ≈ 0.002).

However, the oracle-head continuation baseline is substantially more
consistent across the held-out structural evaluation. L1 tracks the expected
structural reversal on all 14 held-out pairs, whereas the lightweight M
estimator does so on 9 of 14 pairs, with no held-out pair on which M succeeds
and L1 fails.

The activation-conditioned signal is also heterogeneous across structural
categories: role/argument binding, modifier attachment, and spatial relations
are comparatively legible, while comparative relations show systematic
negative alignment and quantifier binding is mixed.

Taken together, these results argue against interpreting multi-token
recoverability as straightforward evidence that a readout has extracted
substantial additional information from activations.

A better interpretation is that multi-token recovery can arise from two
sources:

1. genuinely activation-conditioned structural information, and
2. a strong continuation prior in the frozen language model once a useful
   first-token clue is supplied.

In this controlled pilot, both sources are present, but the continuation
mechanism is the more consistent source of recoverability.

Because the activation-conditioned estimator used here is a lightweight
approximation rather than the fully calibrated multi-token method, the result
should be interpreted as a decomposition result about this experimental setup,
not as a negative conclusion about multi-token J-Lens in general.

## Final research conclusion

The project asked whether a multi-token readout can reflect information already visible in activations, and how much ordinary continuation can recover once the first token is supplied.

The result was not one-sided.

M shows real pair-specific activation-side structural signal beyond permutation controls. But L1 is substantially more consistent across the held-out set: 14/14 expected-direction successes for L1 versus 9/14 for M, with no M-only pair.

So the strongest conclusion is not that activation-side structure is absent. It is that, with this lightweight readout and this dataset, first-token-conditioned continuation explains the tested distinctions more consistently.

This also changes how I would interpret a successful multi-token verbalization. A readout producing the correct phrase does not by itself show that the full phrase was directly recovered from the hidden state. Some of the remaining sequence may be supplied by the model's own continuation behavior.

These results apply to the current lightweight M metric, not to full multi-token J-Lens in general.

# Interpretation complete

The main interpretation, limitations, alternative explanations, and follow-up directions are now fixed.

No frozen Gate 3 result is changed in this section.

# Pre-write-up verification

Before writing the final report, independently recheck the load-bearing statistics from the frozen experiment.

This is verification only. No estimator settings are changed.

## Manual verification

Recompute the headline M and L1 statistics and inspect the underlying pair-level results before finalizing the write-up.

In [2]:
# 5.8a — Recompute headline success counts from saved held-out results

import json
from pathlib import Path

results_dir = Path("/workspace/multi-token-jlens/results")

with open(results_dir / "heldout_m_results.json", "r") as f:
    verify_m = json.load(f)

with open(results_dir / "heldout_l1_results.json", "r") as f:
    verify_l1 = json.load(f)

# M success is defined by positive mean alignment
m_success_by_pair = {
    row["pair_id"]: (row["mean_alignment"] > 0)
    for row in verify_m
}

# L1 success is explicitly stored as boolean
l1_success_by_pair = {
    row["pair_id"]: bool(row["success"])
    for row in verify_l1
}

pair_ids = sorted(m_success_by_pair.keys())

m_success_count = sum(m_success_by_pair[pid] for pid in pair_ids)
l1_success_count = sum(l1_success_by_pair[pid] for pid in pair_ids)

both = sum(
    m_success_by_pair[pid] and l1_success_by_pair[pid]
    for pid in pair_ids
)

l1_only = sum(
    (not m_success_by_pair[pid]) and l1_success_by_pair[pid]
    for pid in pair_ids
)

m_only = sum(
    m_success_by_pair[pid] and (not l1_success_by_pair[pid])
    for pid in pair_ids
)

neither = sum(
    (not m_success_by_pair[pid]) and (not l1_success_by_pair[pid])
    for pid in pair_ids
)

print("M success:", m_success_count, "/", len(pair_ids))
print("L1 success:", l1_success_count, "/", len(pair_ids))
print()
print("Both:", both)
print("L1 only:", l1_only)
print("M only:", m_only)
print("Neither:", neither)

M success: 9 / 14
L1 success: 14 / 14

Both: 9
L1 only: 5
M only: 0
Neither: 0


In [3]:
# 5.8b — Recompute M matched mean and permutation significance

import numpy as np
import json
from pathlib import Path

results_dir = Path("/workspace/multi-token-jlens/results")

with open(results_dir / "heldout_m_results.json", "r") as f:
    verify_m = json.load(f)

matched_means = np.array([
    row["mean_alignment"]
    for row in verify_m
])

matched_mean = matched_means.mean()

print("Recomputed matched M mean:", matched_mean)
print("Positive pairs:", int((matched_means > 0).sum()), "/", len(matched_means))

Recomputed matched M mean: 0.042004271307142856
Positive pairs: 9 / 14


In [4]:
# 5.8c — Inspect saved result files for permutation-null artifacts

from pathlib import Path

results_dir = Path("/workspace/multi-token-jlens/results")

for p in sorted(results_dir.iterdir()):
    print(p.name)

heldout_l1_results.json
heldout_m_results.json
mplus_comparative_robustness.json


In [5]:
# 5.8d — Check whether frozen M permutation objects are still in memory

names_to_check = [
    "m_diff_cache",
    "sweep_layers",
    "perm_pair_means",
    "matched_pair_means",
    "matched_positive_pairs",
]

for name in names_to_check:
    if name in globals():
        obj = globals()[name]
        print(
            f"{name}: FOUND | "
            f"type={type(obj).__name__} | "
            f"len={len(obj) if hasattr(obj, '__len__') else 'n/a'}"
        )
    else:
        print(f"{name}: NOT FOUND")

m_diff_cache: NOT FOUND
sweep_layers: NOT FOUND
perm_pair_means: NOT FOUND
matched_pair_means: NOT FOUND
matched_positive_pairs: NOT FOUND


In [6]:
# 5.8e — Load independently verified M permutation summary

import json
from pathlib import Path

verification_path = Path(
    "/workspace/multi-token-jlens/results/"
    "m_permutation_verification.json"
)

with open(verification_path, "r") as f:
    m_perm_verify = json.load(f)

for k, v in m_perm_verify.items():
    print(f"{k}: {v}")

matched_mean: 0.04200427130230571
null_mean: -0.0020694622421870007
null_median: -0.002591174505104558
null_95th_percentile: 0.017852476586549686
null_max: 0.03935324800039204
empirical_one_sided_p: 0.001996007984031936
n_permutations: 500


In [7]:
# 5.8f — Locate original code for positive-count p and L1 matched/null ratio

from pathlib import Path
import json

nb_path = Path(
    "/workspace/multi-token-jlens/notebooks/03_core_experiment.ipynb"
)

nb = json.loads(nb_path.read_text())

search_terms = [
    "positive count",
    "positive_count",
    "matched_positive",
    "15.8",
    "matched / null",
    "mean abs shift",
    "permutation_mean_abs",
]

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))

    if any(
        term.lower() in source.lower()
        for term in search_terms
    ):
        print(f"\n--- CELL {i} ---")
        print(source[:6000])


--- CELL 75 ---
matched_shifts = np.array([
    r["paired_shift"]
    for r in heldout_l1_results
])

mismatched_shifts = np.array([
    r["paired_shift"]
    for r in l1_mismatch_results
])

print("MATCHED")
print("Mean shift:       ", matched_shifts.mean())
print("Median shift:     ", np.median(matched_shifts))
print("Mean abs shift:   ", np.abs(matched_shifts).mean())

print()

print("MISMATCHED")
print("Mean shift:       ", mismatched_shifts.mean())
print("Median shift:     ", np.median(mismatched_shifts))
print("Mean abs shift:   ", np.abs(mismatched_shifts).mean())

print()

print(
    "Matched > mismatched magnitude:",
    int(
        (
            np.abs(matched_shifts)
            > np.abs(mismatched_shifts)
        ).sum()
    ),
    "/",
    len(matched_shifts)
)

print(
    "Mean magnitude ratio:",
    np.abs(matched_shifts).mean()
    / np.abs(mismatched_shifts).mean()
)

--- CELL 77 ---
import random

random.seed(42)

n_permutations = 20
permutation_mean_abs_shifts = []

## Verification summary

The headline results were independently reproduced.

### M

- expected direction: 9/14
- matched mean alignment: +0.0420
- permutation-null mean: -0.0021
- empirical one-sided p for matched mean: about 0.002
- positive-count p: about 0.15

### L1

- expected direction: 14/14
- matched mean absolute shift: 2.1646
- mismatched mean absolute shift: 0.0931
- permutation-null mean absolute shift: 0.1370

### Directional comparison

- both: 9
- L1 only: 5
- M only: 0
- neither: 0

The verification reproduced the statistics used in the final interpretation. No frozen experiment settings or conclusions were changed.